In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
print("Imports done!!")

Imports done!!


In [2]:
GBP_TO_INR = 105.50
BOOK_SCRAPER_URL = "https://books.toscrape.com/catalogue/category/books"
GENRES_TO_SCRAPE = [
    "travel_2",
    "mystery_3",
    "historical-fiction_4",
    "sequential-art_5",
    "classics_6"
]
RATING_DICT = {
    "One": 1, 
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [3]:
# Function to scrape books for a specific genre - TASK 1
def scrape_books_for_genre(html: str, genre: str):
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception as e:
        print(f"Error parsing HTML for genre {genre}: {e}")
        return []
    all_books = soup.find_all("article", class_="product_pod")
    book_jsons = []
    for book in all_books:
        book_json = {
            "title": book.h3.a["title"],
            "category": genre,
            "price_in_gbp": book.find("p", class_="price_color").text.strip(),
            "star_rating": book.find("p", class_="star-rating").get("class", [])[-1],
            "availability": book.find("p", class_="availability").text.strip()
        }
        book_jsons.append(book_json)
    return book_jsons

def scrape_all_books():
    books = []
    for genre in GENRES_TO_SCRAPE:
        genre_url = f"{BOOK_SCRAPER_URL}/{genre}/index.html"
        print(f"Scraping genre: {genre} from URL: {genre_url}")
        response = requests.request("GET", genre_url)
        if response.status_code == 200:
            books.extend(scrape_books_for_genre(response.text, genre))
    print(f"Total books scraped: {len(books)}")
    return books

In [4]:
# Functions to clean the data

def get_rating(star_rating):
    try:
        return RATING_DICT.get(star_rating, 0)
    except Exception as e:
        print(f"Error in get_rating: {e}")
        return None

def clean_transform_data(books):
    print("Cleaning and transforming data...")
    print("converting the scraped books to a pandas dataframe")
    df = pd.DataFrame(books,)

    print("before cleaning...")
    print(df.info())
    print(df.head())

    print("-" * 30)

    print("dropping the rows containing null values in title column")
    df["title"].dropna(inplace=True)

    print("remooving the number at the end of catregories")
    df["category"] = df["category"].str.split("_").str[0]

    print("making price in gbp a number column and imputing null values with mean")
    df["price_in_gbp"] = df["price_in_gbp"].str.replace("Â£", "").astype(float, errors='raise')
    df["price_in_gbp"] = df["price_in_gbp"].fillna(df["price_in_gbp"].mean())

    print("making star rating a number column and imputing null values with median")
    df["star_rating"] = df["star_rating"].apply(get_rating).astype(int, errors='raise')
    df["star_rating"] = df["star_rating"].fillna(df["star_rating"].median())

    print("making availability a boolean column")
    df["availability"] = df["availability"].apply(lambda x: 1 if x == "In stock" else 0).astype(bool)
    print("-" * 30)

    print("after cleaning...")
    print(df.info())
    print(df.head())
    return df

def add_price_in_inr(books):
    print("adding the column price in inr...")
    print(f"Conversion rate: 1 GBP = {GBP_TO_INR} INR")
    books['price_in_inr'] = books['price_in_gbp'] * GBP_TO_INR
    print(books.head())
    return books

In [5]:
# functions to insert data into database

def create_db():
    print("Creating database books...")
    conn = sqlite3.connect("books.db")
    cursor = conn.cursor()
    cursor.execute('DROP TABLE IF EXISTS books')
    cursor.execute('DROP TABLE IF EXISTS categories')

    print('Creating table - categories')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS categories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE
        )
    ''')

    print('Creating table - books')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS books (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            category INTEGER NOT NULL,
            price_in_gbp REAL NOT NULL,
            star_rating INTEGER NOT NULL,
            availability BOOLEAN NOT NULL,
            price_in_inr REAL NOT NULL,
            FOREIGN KEY (category) REFERENCES categories(id)
        )
    ''')
    print("Database created successfully.")
    return cursor


def add_books_to_db(cursor, books, category_mapping):
    print("clearing the existing books in the database...")
    cursor.execute('DELETE FROM books')
    print("Adding books to the database...")
    books_to_insert = []
    for book in books.to_dict(orient="records"):
        category_id = category_mapping[book["category"]]
        books_to_insert.append((
            book["title"],
            category_id,
            book["price_in_gbp"],
            book["star_rating"],
            book["availability"],
            book["price_in_inr"]
        ))
    cursor.executemany(
        '''
        INSERT INTO books 
        (title, category, price_in_gbp, star_rating, availability, price_in_inr) 
        VALUES (?, ?, ?, ?, ?, ?)
        ''', books_to_insert)
    print(f"Inserted {len(books_to_insert)} books into the database.")


def create_categories(cursor, books):
    print("creating categories...")
    unique_categories = books["category"].unique()
    category_mapping = {}
    categories_to_insert = []
    for i, category in enumerate(unique_categories):
        categories_to_insert.append((i, category))
        category_mapping[category] = i
    cursor.execute('DELETE FROM categories')
    cursor.executemany('INSERT INTO categories (id, name) VALUES (?, ?)', categories_to_insert)
    print(f"Inserted {len(categories_to_insert)} categories into the database.")
    return category_mapping

In [6]:
# functions to show stats

def show_stats_using_sqlite(cursor):
    '''
    Query 1 - how many books are there having more than 4* rating?
    Query 2 - what is the average price of books in GBP and INR in each category?
    Query 3 - in what categories are the top 10 expensive books fall?
    Query 4 - what is the most loved genre of books based on the average rating?
    Query 5 - how many books are there between 20 and 30 GBP price range?
    '''
    print("Executing SQL queries to show stats...")
    print("=" * 30)
    # Query 1
    query1 = "SELECT COUNT(id) FROM books WHERE star_rating >= 4"
    query1_result = cursor.execute(query1).fetchone()[0]
    print(f"Query 1: {query1}\nResult: \nNumber of books with more than 4 stars: {query1_result}")

    # Query 2
    query2 = """
    SELECT c.name, AVG(b.price_in_gbp) as avg_price_gbp, AVG(b.price_in_inr) as avg_price_inr
    FROM books b
    JOIN categories c ON b.category = c.id
    GROUP BY c.name
    """
    query2_result = cursor.execute(query2).fetchall()
    print("-" * 20)
    print(f"Query 2: {query2}\nResult: \nAverage price of books in GBP and INR for each category:")
    for row in query2_result:
        print(f"Category: {row[0]}, Average Price (GBP): {row[1]:.2f}, Average Price (INR): {row[2]:.2f}")
    print("-" * 20)

    # Query 3
    query3 = """
    SELECT DISTINCT(c.name)
    FROM books b
    JOIN categories c ON b.category = c.id
    ORDER BY b.price_in_gbp DESC
    LIMIT 10
    """
    query3_result = cursor.execute(query3).fetchall()
    print(f"Query 3: {query3}\nResult: \nCategories of the top 10 expensive books:")
    for row in query3_result:
        print(f" - {row[0]}")
    print("-" * 20)

    # Query 4
    query4 = """
    SELECT c.name, AVG(b.star_rating) as avg_rating
    FROM books b
    JOIN categories c ON b.category = c.id
    GROUP BY c.name
    ORDER BY avg_rating DESC
    LIMIT 1
    """
    query4_result = cursor.execute(query4).fetchone()
    print(f"Query 4: {query4}\nResult: \nMost loved genre of books: {query4_result[0]} with average rating: {query4_result[1]:.2f}")
    print("-" * 20)

    # Query 5
    query5 = """
    SELECT COUNT(id) FROM books WHERE price_in_gbp BETWEEN 20 AND 30
    """
    query5_result = cursor.execute(query5).fetchone()[0]
    print(f"Query 5: {query5}\nResult: \nNumber of books priced between 20 and 30 GBP: {query5_result}")
    print("=" * 30)

def show_stats_using_pandas(cursor):
    print("Reading Query 2 and Query 3 as DataFrames using pandas...")

    # Query 2
    query2 = """
    SELECT c.name, AVG(b.price_in_gbp) as avg_price_gbp, AVG(b.price_in_inr) as avg_price_inr
    FROM books b
    JOIN categories c ON b.category = c.id
    GROUP BY c.name
    """
    query2_df = pd.read_sql(query2, cursor.connection)
    print("-" * 20)
    print("Query 2 Result:")
    print(query2_df)

    # Query 3
    query3 = """
    SELECT DISTINCT(c.name)
    FROM books b
    JOIN categories c ON b.category = c.id
    ORDER BY b.price_in_gbp DESC
    LIMIT 10
    """
    query3_df = pd.read_sql(query3, cursor.connection)
    print("-" * 20)
    print("Query 3 Result:")
    print(query3_df) 

    print("-" * 20)
    print("Getting the books table as df...")
    books = pd.read_sql("SELECT * FROM books", cursor.connection)
    print("Getting the categories table as df...")
    categories = pd.read_sql("SELECT * FROM categories", cursor.connection)
    merged_df = pd.merge(books, categories, left_on="category", right_on="id", suffixes=("_book", "_category"))
    print("Merged DataFrame:")
    print(merged_df.head())
    print("-" * 20)
    print("executing query2 using pandas...")
    query2_pandas = merged_df.groupby("name").agg(
        avg_price_gbp=("price_in_gbp", "mean"),
        avg_price_inr=("price_in_inr", "mean")
    )
    print("Query 2 Result (using pandas):")
    print(query2_pandas)

In [7]:
# Main function which calls all the functions

def main():
    print("###################################### TASK 1 ######################################")
    books = scrape_all_books()

    print("\n###################################### TASK 2 ######################################")
    transformed_books = clean_transform_data(books)

    print("\n###################################### TASK 3 ######################################")
    transformed_books = add_price_in_inr(transformed_books)

    print("\n###################################### TASK 4 ######################################")
    cursor = create_db()
    category_mapping = create_categories(cursor, transformed_books)
    add_books_to_db(cursor, transformed_books, category_mapping)

    print("\n###################################### TASK 5 ######################################")
    show_stats_using_sqlite(cursor)

    print("\n###################################### TASK 6 ######################################")
    show_stats_using_pandas(cursor)

    # Close the DB Connection
    cursor.close()
    cursor.connection.close()

In [8]:
if __name__ == "__main__":
    main()

###################################### TASK 1 ######################################
Scraping genre: travel_2 from URL: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Scraping genre: mystery_3 from URL: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Scraping genre: historical-fiction_4 from URL: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Scraping genre: sequential-art_5 from URL: https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Scraping genre: classics_6 from URL: https://books.toscrape.com/catalogue/category/books/classics_6/index.html
Total books scraped: 90

###################################### TASK 2 ######################################
Cleaning and transforming data...
converting the scraped books to a pandas dataframe
before cleaning...
<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column        Non-Null C